# DS 227 · Week 9 — Data journalism
## From dataset to defensible claim

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 6 — Code on these slides runs in the browser


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 14 — The flood data, loaded from the course API


Expected output:

```
(34079, 15)
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
resp = requests.get(url, timeout=60)
df   = pd.read_csv(io.StringIO(resp.text))

print(df.shape)

### Slide 15 — A column of zeros does not mean "unpaid"


Expected output:

```
[0]
0
1.17
```


In [ ]:
print(df.amount_paid.unique())
print(df.amount_paid.sum())

# the tempting "scandal" number, in PHP trillions
print(round(df[df.status == "Completed"].budget.sum() / 1e12, 2))

### Slide 19 — Ten students, two sections


Expected output:

```
student  study_hours section  score  passed
0       A            1       A     58   False
1       B            6       A     84    True
2       C            2       B     61   False
3       D            8       B     96    True
4       E            3       A     66   False
5       F            7       B     90    True
6       G            2       A     60   False
7       H            9       B     98    True
8       I            4       A     72   False
9       J            5       B     80    True
```


In [ ]:
students = pd.DataFrame({
    "student": list("ABCDEFGHIJ"),
    "study_hours": [1, 6, 2, 8, 3, 7, 2, 9, 4, 5],
    "section": ["A", "A", "B", "B", "A", "B", "A", "B", "A", "B"],
    "score": [58, 84, 61, 96, 66, 90, 60, 98, 72, 80],
})
students["passed"] = students["score"] >= 75

print(students)

### Slide 20 — Turn the question into a comparison


**Exercise.**

1. Predict the average study hours of the students who passed.
2. Run the cell.
3. Change the pass mark to 85 (students["score"] >= 85 on the table slide) and run again. How do the groups change?


In [ ]:
# students: the ten students from the table slide
print(students.groupby("passed")["study_hours"].agg(
    ["mean", "min", "max", "count"]))

### Slide 22 — A figure needs a reference for comparison


Expected output:

```
1.6
11
145.5
```


In [ ]:
print(round(df.budget.sum() / 1e12, 2))        # PHP trillion, 2016-2026
print(df.year.nunique())                       # years

print(round(df.budget.sum() / 1e9 / 11, 1))    # PHP billion per year

### Slide 24 — Counts versus per-capita rates


**Exercise.**

1. Predict which place has more reports per 100,000 people.
2. Run the cell.
3. Change Town B’s reports from 40 to 30. Does it still lead?


In [ ]:
# made-up example: flood reports and population
reports = pd.Series({"City A": 500, "Town B": 40})
people  = pd.Series({"City A": 1_000_000, "Town B": 50_000})

print(reports / people * 100_000)      # reports per 100,000 people

### Slide 25 — Context for a comparison


Expected output:

```
region
Central Office    242.0
Region IV-B        67.1
Region XIII        61.0
Region I           31.2
Name: per_project, dtype: float64
```


In [ ]:
g = df.groupby("region").agg(
    n=("contract_id", "count"),
    total=("budget", "sum"))
g["per_project"] = g.total / g.n
top = (g.per_project / 1e6).round(1).sort_values(ascending=False)
print(top.iloc[[0, 1, 2, -1]])     # PHP millions

### Slide 26 — Charting the contrast honestly


In [ ]:
# students: the ten students from the table slide
import matplotlib.pyplot as plt

means = students.groupby("passed")["study_hours"].mean()

fig, ax = plt.subplots()
ax.bar(["failed", "passed"], means)
ax.set_ylabel("Avg study hours")
plt.show()

### Slide 28 — The finding as one sentence


Expected output:

```
passed avg 7.0h vs failed avg 2.4h (gap 4.6h, n=10)
```


In [ ]:
# students: the ten students from the table slide
g = students.groupby("passed")["study_hours"].mean()
print(f"passed avg {g[True]:.1f}h vs failed avg {g[False]:.1f}h "
      f"(gap {g[True] - g[False]:.1f}h, n={len(students)})")

### Slide 32 — Checking for a confounder


Expected output:

```
section
A    0.2
B    0.8
Name: passed, dtype: float64
```


In [ ]:
# students: the ten students from the table slide
print(students.groupby("section")["passed"].mean())

### Slide 33 — Robustness under a different framing


**Exercise.**

1. Predict the pass rate of the students who studied 5 hours or more.
2. Run the cell.
3. Change the cut-off to 4 hours (>= 4). Does the finding survive?


In [ ]:
# students: the ten students from the table slide
students["high_study"] = students["study_hours"] >= 5
print(students.groupby("high_study")["passed"].mean())

### Slide 36 — 131 firms, 1,263 projects, ₱75.2 billion


Expected output:

```
131
1263
75.2
0.047
```


In [ ]:
rev = df[df.contractor.str.contains("REVOKED", na=False)]

print(rev.contractor.nunique())               # firms
print(len(rev))                               # projects
print(round(rev.budget.sum() / 1e9, 1))       # PHP billions
print(round(rev.budget.sum() / df.budget.sum(), 3))   # share of all money

### Slide 37 — 4.7% of the money compared with the base rate


Expected output:

```
0.027
0.037
0.047
```


In [ ]:
# rev: the projects of revoked firms (previous slide)
print(round(rev.contractor.nunique() / df.contractor.nunique(), 3))   # share of firms
print(round(len(rev) / len(df), 3))                                   # share of projects
print(round(rev.budget.sum() / df.budget.sum(), 3))                   # share of money

### Slide 39 — Licence status at export time only


Expected output:

```
ST. TIMOTHY CONSTRUCTION CORPORATION ([REVOKED] 39196)
['start_date', 'completion_date']
```


In [ ]:
# how a revoked licence appears in the data
print(rev.contractor.iloc[0])

# which columns hold a date?
print([c for c in df.columns if "date" in c])

### Slide 42 — Concentration: a claim with verifiable numbers


Expected output:

```
4841
0.276
0.1
```


In [ ]:
c = (df.groupby("contractor").budget.sum()
       .sort_values(ascending=False))
print(len(c))                                  # contractors

print(round(c.head(48).sum() / c.sum(), 3))    # the top 1%
print(round(c.head(10).sum() / c.sum(), 3))    # the top 10

### Slide 44 — Terminated projects: 131, or 0.4%


Expected output:

```
status
Completed          80.8
On-Going           16.0
For Procurement     2.6
Terminated          0.4
Not Yet Started     0.3
Name: proportion, dtype: float64
```


In [ ]:
print((df.status.value_counts(normalize=True) * 100).round(1))

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
